Teacher Full CNNLSTM model 
Student Full Transformer model
two conditions: cross_domain: train set: room 1, test set1: room 2, test set2: room3
cross_environment: train set: room 1, test set: room 2+3

In [ ]:
import os
import random
import time
import numpy as np
import tensorflow as tf
import gc
import keras
from tensorflow.keras.layers import Input, Dense, Dropout, Layer, Conv2D, MaxPooling2D,  Bidirectional, LSTM, GRU
from tensorflow.keras.layers import TimeDistributed, Flatten, Softmax, BatchNormalization, MultiHeadAttention, LayerNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler, ReduceLROnPlateau
from tensorflow.keras.regularizers import  l2
from tensorflow.keras.losses import CategoricalCrossentropy
import keras.backend as K
from sklearn.metrics import (
    precision_score, recall_score, f1_score,accuracy_score,
    cohen_kappa_score, classification_report, confusion_matrix
)
from sklearn.model_selection import train_test_split
import tf2onnx
import onnx
import onnxruntime as ort 
from onnxruntime.quantization.shape_inference import quant_pre_process
from onnxruntime.quantization import (
    quantize_static,
    CalibrationDataReader,
    QuantType,
    QuantFormat,
    CalibrationMethod
)
import hashlib
import time as _time
from scipy.stats import wilcoxon, ttest_rel

In [ ]:
# Teacher model
def assemble_model(input_shape, n_class):
    model_input = Input(shape=input_shape, dtype='float32', name='name_model_input')

    # Feature extraction part
    x = TimeDistributed(Conv2D(32, kernel_size=(5, 5), activation='relu', data_format='channels_last'))(model_input) 
    x = TimeDistributed(MaxPooling2D(pool_size=(2, 2)))(x)
    x = TimeDistributed(Flatten())(x) 
    x = TimeDistributed(Dense(256, activation='gelu'))(x)
    x = Bidirectional(GRU(128, return_sequences=True))(x)
    x = Dropout(0.2)(x)
    x = LSTM(128, return_sequences=True)(x)
    x = Dropout(0.2)(x)

    # Output part
    x = Flatten()(x)
    model_output = Dense(n_class, activation='softmax', name='name_model_output')(x) 

    # Model compiling
    model = Model(inputs=model_input, outputs=model_output)
    # Define the loss function with label smoothing
    loss_function = CategoricalCrossentropy(label_smoothing=0.1)
    model.compile(optimizer=keras.optimizers.RMSprop(learning_rate=0.001),
                  loss=loss_function,
                  metrics=['accuracy'])
    return model

# Instantiate the model with the desired number of layers
model_1 = assemble_model(input_shape=(36, 20, 20, 1), n_class=6)

# Student Model
# Create positional encoding
def positional_encoding(seq_len, d_model):
    angles = 1 / np.power(10000., (2 * (np.arange(d_model)[np.newaxis, :] // 2)) / np.float32(d_model))
    positional_encoding = np.zeros((seq_len, d_model))
    positional_encoding[:, 0::2] = np.sin(np.dot(np.arange(seq_len)[:, np.newaxis], angles[:, 0::2]))
    positional_encoding[:, 1::2] = np.cos(np.dot(np.arange(seq_len)[:, np.newaxis], angles[:, 1::2]))
    positional_encoding = positional_encoding[np.newaxis, ...]
    return tf.cast(positional_encoding, dtype=tf.float32)

#Create MultiHeadAttention Encoder
class ScaledDotProductAttention(Layer):
    def __init__(self, dropout_ratio=0.1, **kwargs):
        super(ScaledDotProductAttention,
              self).__init__(**kwargs)
        self.dropout_ratio = dropout_ratio  
        self.dropout       = Dropout(dropout_ratio)

    def call(self, q, k, v,
             mask=None, training=False):
        matmul_qk = tf.matmul(q, k, transpose_b=True)
        dk = tf.cast(tf.shape(k)[-1], tf.float32)
        scaled_attention_logits = (
            matmul_qk / tf.math.sqrt(dk)
        )

        if mask is not None:
            scaled_attention_logits += (mask * -1e9)

        attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)
        attention_weights = self.dropout(attention_weights, training=training)
        output = tf.matmul(attention_weights, v)
        return output

    def get_config(self):
        config = super().get_config()
        config.update({
            'dropout_ratio': self.dropout_ratio
        })
        return config


# MultiHeadAttention
class MultiHeadAttention(Layer):
    def __init__(self, d_model, num_heads,
                 dropout_ratio=0.1, **kwargs):
        super(MultiHeadAttention,
              self).__init__(**kwargs)
        self.num_heads     = num_heads
        self.d_model       = d_model
        self.dropout_ratio = dropout_ratio

        assert d_model % self.num_heads == 0

        self.depth = d_model // self.num_heads

        self.wq    = Dense(d_model)
        self.wk    = Dense(d_model)
        self.wv    = Dense(d_model)

        # Create ONCE in __init__, not in call()
        self.attention = ScaledDotProductAttention(dropout_ratio)
        self.dropout   = Dropout(dropout_ratio)
        self.dense     = Dense(d_model)

    def split_heads(self, x, batch_size):
        x = tf.reshape(
            x,
            (batch_size, -1,
             self.num_heads, self.depth)
        )
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, q, k, v,
             mask=None, training=False):
        batch_size = tf.shape(q)[0]

        q = self.wq(q)
        k = self.wk(k)
        v = self.wv(v)
        q = self.split_heads(q, batch_size)
        k = self.split_heads(k, batch_size)
        v = self.split_heads(v, batch_size)
        scaled_attention = self.attention( q, k, v, mask, training)
        scaled_attention = tf.transpose(scaled_attention, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(scaled_attention,(batch_size, -1, self.d_model))
        output = self.dense(concat_attention)
        return output

    def get_config(self):
        config = super().get_config()
        config.update({
            'd_model'      : self.d_model,
            'num_heads'    : self.num_heads,
            'dropout_ratio': self.dropout_ratio
        })
        return config


# TransformerMultiHeadAttentionEncoderLayer
class TransformerMultiHeadAttentionEncoderLayer(Layer):
    def __init__(self, ff_dim,
                 dropout_ratio, **kwargs):
        super(
            TransformerMultiHeadAttentionEncoderLayer,
            self
        ).__init__(**kwargs)
        self.ff_dim        = ff_dim
        self.dropout_ratio = dropout_ratio
        self.dropout1 = Dropout(dropout_ratio)
        self.dropout2 = Dropout(dropout_ratio)

    def build(self, input_shape):
        d_model = input_shape[-1]

        self.attention   = MultiHeadAttention(
            d_model       = d_model,
            num_heads     = 4,
            dropout_ratio = self.dropout_ratio
        )
        self.layer_norm1 = LayerNormalization(epsilon=1e-6)
        self.dense1      = Dense(
            self.ff_dim,
            activation         = 'gelu',
            kernel_regularizer = l2(0.01)
        )
        self.dense2      = Dense(
            d_model,
            kernel_regularizer = l2(0.01)
        )
        self.layer_norm2 = LayerNormalization(epsilon=1e-6)
        super(
            TransformerMultiHeadAttentionEncoderLayer,
            self
        ).build(input_shape)

    def call(self, inputs, training=False):
        seq_len = inputs.shape[1]
        d_model = inputs.shape[2]

        pos_encoding = positional_encoding(seq_len, d_model)
        x = inputs + pos_encoding

        attn_output = self.attention(x, x, x, training=training)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layer_norm1(inputs + attn_output)

        ffn_output = self.dense1(out1)
        ffn_output = self.dense2(ffn_output)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layer_norm2(out1 + ffn_output)

    def get_config(self):
        config = super().get_config()
        config.update({
            'ff_dim'       : self.ff_dim,
            'dropout_ratio': self.dropout_ratio
        })
        return config

    
def transformer_model(input_shape, n_class, num_layers=2):
    model_input = Input(shape=input_shape, dtype='float32', name='name_model_input')
    
    # Feature extraction part
    x = TimeDistributed(Flatten())(model_input)
    x = TimeDistributed(Dense(128, activation='gelu'))(x)
    x = TimeDistributed(Dropout(0.5))(x)
    
    # Transformer encoder stack
    for _ in range(num_layers):
        x = TransformerMultiHeadAttentionEncoderLayer(ff_dim=256, dropout_ratio=0.1)(x)
    
    # Output part
    x = Flatten()(x)
    model_output = Dense(n_class, activation='softmax', name='name_model_output')(x)
    
    # Model compiling
    model = Model(inputs=model_input, outputs=model_output)
    # Define the loss function with label smoothing
    loss_function = CategoricalCrossentropy(label_smoothing=0.1)
    model.compile(optimizer=keras.optimizers.RMSprop(learning_rate=0.0005),
                  loss = loss_function,
                  metrics=['accuracy'])
    return model

# Instantiate the model with the desired number of layers
model_2 = transformer_model(input_shape=(36, 20, 20, 1), n_class=6, num_layers=2)

In [ ]:
# Pre-processing the models for onnx
def _file_md5(filepath, chunk_bytes=1024 * 1024):
    h = hashlib.md5()
    with open(filepath, 'rb') as f:
        while True:
            chunk = f.read(chunk_bytes)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def preprocess_onnx(fp32_path, preprocessed_path):
    sidecar_path = preprocessed_path + '.src_md5'
    def should_reprocess():
        if not os.path.exists(preprocessed_path):
            return True, "preprocessed file not found"

        fp32_mtime   = os.path.getmtime(fp32_path)
        preproc_mtime = os.path.getmtime(preprocessed_path)
        if fp32_mtime >= preproc_mtime:
            return True, (
                f"fp32 source is newer than preprocessed "
                f"(fp32={fp32_mtime:.0f} "
                f">= preproc={preproc_mtime:.0f})")

        if not os.path.exists(sidecar_path):
            return True, "MD5 sidecar not found"
        with open(sidecar_path, 'r') as f:
            stored_md5 = f.read().strip()

        current_md5 = _file_md5(fp32_path)
        if stored_md5 != current_md5:
            return True, (
                f"fp32 source MD5 changed "
                f"(stored={stored_md5[:8]}... "
                f"current={current_md5[:8]}...)")

        return False, "cache valid"
    reprocess, reason = should_reprocess()

    if not reprocess:
        print(f"    [PreProcess] Cache hit — skipping "
              f"({reason}): "
              f"{os.path.basename(preprocessed_path)}")
        return preprocessed_path

    print(f"    [PreProcess] Running shape inference + "
          f"graph optimization...")
    print(f"    [PreProcess] Reason: {reason}")

    if os.path.exists(preprocessed_path):
        os.remove(preprocessed_path)
        print(f"    [PreProcess] Deleted stale: "
              f"{os.path.basename(preprocessed_path)}")
    if os.path.exists(sidecar_path):
        os.remove(sidecar_path)

    quant_pre_process(
        input_model_path    = fp32_path,
        output_model_path   = preprocessed_path,
        skip_optimization   = False,
        skip_onnx_shape     = False,
        skip_symbolic_shape = False,
        verbose             = 0)
    preprocessed_model = onnx.load(preprocessed_path)
    onnx.checker.check_model(preprocessed_model)
    src_md5 = _file_md5(fp32_path)
    with open(sidecar_path, 'w') as f:
        f.write(src_md5)
    size_mb = os.path.getsize(preprocessed_path) / (1024**2)
    print(f"    [✓] Preprocessed : {size_mb:.4f} MB")
    print(f"    [✓] MD5 sidecar  : {src_md5[:16]}...")
    return preprocessed_path

class CalibrationReader(CalibrationDataReader):
    def __init__(self,
                 calibration_data,
                 calibration_labels,  
                 onnx_path,
                 n_samples  = 300,
                 seed       = 42):
        model             = onnx.load(onnx_path)
        graph_input_names = [inp.name for inp in model.graph.input]
        initializer_names = {
            init.name
            for init in model.graph.initializer}
        true_inputs = [
            name for name in graph_input_names
            if name not in initializer_names]
        if len(true_inputs) == 0:
            raise ValueError("incorrectly. Re-export with tf2onnx.")
        self.input_name = true_inputs[0]
        print(f"    [Calibration] True input node  : "
              f"'{self.input_name}'")
        print(f"    [Calibration] Total graph inputs: "
              f"{len(graph_input_names)} "
              f"({len(graph_input_names) - len(true_inputs)}"
              f" are initializers)")

        true_input_obj = None
        for inp in model.graph.input:
            if inp.name == self.input_name:
                true_input_obj = inp
                break

        if true_input_obj is not None:
            model_input_shape = [
                dim.dim_value
                for dim in
                true_input_obj.type.tensor_type.shape.dim]
            expected_shape = [
                d for d in model_input_shape[1:]
                if d > 0   ]
            data_shape = list(calibration_data.shape[1:])
            if expected_shape and len(expected_shape) == len(data_shape):
                if data_shape != expected_shape:
                    raise ValueError(
                        f"Shape mismatch.\n"
                        f"  Calibration data : {data_shape}\n"
                        f"  Model expects    : {expected_shape}\n"
                        f"  Check your data preprocessing "
                        f"or export spec.")
                print(f"    [Calibration] Shape verified  : "
                      f"{data_shape} ✓")
            else:
                print(f"    [Calibration] Shape check     : "
                      f"skipped (dynamic dims in model)")
        else:
            print(f"    [Calibration] Shape check     : "
                  f"skipped (input object not found)")
        rng = np.random.RandomState(seed)
        if calibration_labels is not None:
            if calibration_labels.ndim == 2:
                class_indices = np.argmax(calibration_labels, axis=1)
            else:
                class_indices = calibration_labels.astype(int)
            classes        = np.unique(class_indices)
            n_classes      = len(classes)
            n_per_class    = max(
                1,
                n_samples // n_classes
            )
            selected_idx   = []
            for cls in classes:
                cls_mask = np.where(
                    class_indices == cls
                )[0]
                n_take   = min(n_per_class, len(cls_mask))
                chosen   = rng.choice(
                    cls_mask,
                    size    = n_take,
                    replace = False)
                selected_idx.extend(chosen.tolist())
            remaining = list(
                set(range(len(calibration_data)))
                - set(selected_idx))
            shortfall = n_samples - len(selected_idx)
            if shortfall > 0 and len(remaining) > 0:
                extra = rng.choice(
                    remaining,
                    size    = min(shortfall, len(remaining)),
                    replace = False)
                selected_idx.extend(extra.tolist())
            selected_idx = np.array(selected_idx)
            print(f"    [Calibration] Stratified sampling:")
            for cls in classes:
                cls_count = np.sum(
                    class_indices[selected_idx] == cls)
                print(f"      Class {cls}: "
                      f"{cls_count} samples")
                if cls_count < 5:
                    print(
                        f"  WARNING: Class {cls} has "
                        f"only {cls_count} calibration "
                        f"samples. Scale/zero-point for "
                        f"this class will be unreliable. "
                        f"Increase N_CALIB_SAMPLES."
                    )

        else:
            print(f"    [Calibration] No labels provided, "
                  f"using random sampling")
            n = min( n_samples, len(calibration_data))
            selected_idx = rng.choice(
                len(calibration_data),
                size    = n,
                replace = False)

        self.data  = calibration_data[
            selected_idx
        ].astype(np.float32)
        self.index = 0
        print(f"    [Calibration] Total selected  : "
              f"{len(self.data)} samples")
        print(f"    [Calibration] Sample shape    : "
              f"{self.data[0].shape}")

    def get_next(self):
        if self.index >= len(self.data):
            return None
        sample  = self.data[
            self.index : self.index + 1
        ]
        self.index += 1
        return {self.input_name: sample}

    def rewind(self):
        self.index = 0
        print(f"    [Calibration] ↺ Rewound "
              f"(ORT pass 2 starting)")

def verify_quantization(fp32_path, quant_path, quant_type):
    fp32_model  = onnx.load(fp32_path)
    quant_model = onnx.load(quant_path)
    fp32_op_counts = {}
    for node in fp32_model.graph.node:
        op = node.op_type
        fp32_op_counts[op] = fp32_op_counts.get(op, 0) + 1
    quant_op_counts = {}
    for node in quant_model.graph.node:
        op = node.op_type
        quant_op_counts[op] = quant_op_counts.get(op, 0) + 1
    quant_proof_ops = [
        'QuantizeLinear',
        'DequantizeLinear',
        'QLinearConv',
        'QLinearMatMul',
        'MatMulInteger',
        'ConvInteger',]

    rnn_ops = ['LSTM', 'GRU', 'RNN']
    print(f"\n    [Verify] Quantization node counts:")
    total_quant_nodes = 0
    for op in quant_proof_ops:
        count = quant_op_counts.get(op, 0)
        if count > 0:
            print(f"      {op:<28}: {count}")
            total_quant_nodes += count
    print(f"\n    [Verify] Recurrent ops (expected FP32):")
    for op in rnn_ops:
        fp32_count  = fp32_op_counts.get(op, 0)
        quant_count = quant_op_counts.get(op, 0)
        if fp32_count > 0:
            print(f"      {op:<10}: {fp32_count} in FP32 → "
                  f"{quant_count} in quantized "
                  f"(stays FP32, expected)")
    compute_ops   = ['Conv', 'MatMul', 'Gemm']
    fp32_compute  = sum(fp32_op_counts.get(op, 0) for op in compute_ops)
    quant_compute = sum(quant_op_counts.get(op, 0) for op in compute_ops)
    quantized_compute = fp32_compute - quant_compute

    if fp32_compute > 0:
        pct = quantized_compute / fp32_compute * 100
        print(f"\n    [Verify] Compute ops quantized: "
              f"{quantized_compute}/{fp32_compute} "
              f"({pct:.1f}%)")
    if total_quant_nodes == 0:
        raise RuntimeError(
            f"QUANTIZATION FAILED SILENTLY. "
            f"Zero QuantizeLinear/DequantizeLinear nodes "
            f"found in {quant_path}. "
            f"The model is still FP32. "
            f"Check pre-processing step."
        )

    print(f" Quantization verified: "
          f"{total_quant_nodes} Q/DQ nodes present")
    return quant_op_counts

In [ ]:
# Export to PTQ
N_CALIB_SAMPLES = 300
def export_to_onnx(keras_model,
                   save_path,
                   opset=17):
    spec = [tf.TensorSpec(
        shape = (None, 36, 20, 20, 1), 
        dtype = tf.float32,
        name  = 'model_input'
    )]
    tf2onnx.convert.from_keras(
        keras_model,
        input_signature = spec,
        opset           = opset,
        output_path     = save_path)
    onnx.checker.check_model(save_path)
    size_mb = os.path.getsize(save_path) / (1024**2)
    print(f"    [✓] FP32: {size_mb:.4f} MB")
    return size_mb


# Static Quantization
def run_static_ptq(fp32_path,
                   save_path,
                   calibration_data,
                   calibration_labels = None,
                   quant_type='int8',
                   calib_seed=42,
                   n_calib_samples=N_CALIB_SAMPLES,
                   per_channel=True):

    if quant_type not in ('int8', 'int16'):
        raise ValueError(
            f"quant_type must be 'int8' or 'int16', "
            f"got '{quant_type}'")

    print(f"\n    [Static PTQ {quant_type.upper()}] Starting...")
    print(f"    [Static PTQ {quant_type.upper()}] "
          f"per_channel={per_channel}, "
          f"calib_samples={n_calib_samples}, "
          f"method=Percentile")

    preprocessed_path = fp32_path.replace(
        '_fp32.onnx',
        '_fp32_preprocessed.onnx')
    preprocess_onnx(fp32_path, preprocessed_path)
    calib_reader = CalibrationReader(
        calibration_data = calibration_data,
        calibration_labels = calibration_labels,
        onnx_path        = preprocessed_path,
        n_samples        = n_calib_samples,
        seed             = calib_seed)
    if quant_type == 'int8':
        act_type  = QuantType.QInt8
        w_type    = QuantType.QInt8
        quant_fmt = QuantFormat.QDQ

        extra_options = {
            'WeightSymmetric'          : True,
            'ActivationSymmetric'      : False,
            'CalibPercentile'          : 99.99,
            'EnableSubgraph'           : False,}

    elif quant_type == 'int16':
        act_type  = QuantType.QInt16
        w_type    = QuantType.QInt8
        quant_fmt = QuantFormat.QDQ
        extra_options = {
            'WeightSymmetric'          : True,
            'ActivationSymmetric'      : False,
            'CalibPercentile'          : 99.99,
            'EnableSubgraph'           : False,}

    quantize_static(
        model_input             = preprocessed_path,
        model_output            = save_path,
        calibration_data_reader = calib_reader,
        quant_format            = quant_fmt,
        activation_type         = act_type,
        weight_type             = w_type,
        calibrate_method        = CalibrationMethod.Percentile,
        per_channel             = per_channel,
        reduce_range            = False,
        extra_options           = extra_options,)

    # ── Mandatory verification ────────────────────────────
    verify_quantization(
        fp32_path  = preprocessed_path,
        quant_path = save_path,
        quant_type = quant_type
    )
    size_mb = os.path.getsize(save_path) / (1024**2)
    fp32_mb = os.path.getsize(fp32_path)  / (1024**2)
    reduction = (1 - size_mb / fp32_mb) * 100
    print(f"\n    [✓] Static {quant_type.upper()} complete")
    print(f"    FP32 size : {fp32_mb:.4f} MB")
    print(f"    Quant size: {size_mb:.4f} MB")
    print(f"    Reduction : {reduction:.1f}%")

    return size_mb


#  Evaluate ONNX
def evaluate_onnx(onnx_path,
                  test_data,
                  test_labels,
                  label_name,
                  n_warmup        = 10,
                  latency_samples = 200,
                  batch_size      = 32):

    if not os.path.exists(onnx_path):
        print(f"      [SKIP] Not found: "
              f"{os.path.basename(onnx_path)}")
        return None
    try:
        sess_options = ort.SessionOptions()
        sess_options.intra_op_num_threads = 1
        sess_options.inter_op_num_threads = 1
        sess = ort.InferenceSession(
            onnx_path,
            sess_options = sess_options,
            providers  = ['CPUExecutionProvider']
        )
    except Exception as e:
        print(f" [ERROR] Session load failed: {e}")
        return None
    input_name = sess.get_inputs()[0].name
    n_total  = len(test_data)
    n_warmup_actual = min(n_warmup, n_total)
    for i in range(n_warmup_actual):
        s = test_data[i : i+1].astype(np.float32)
        sess.run(None, {input_name: s})
    n_lat      = min(latency_samples, n_total)
    lat_times  = []
    for i in range(n_lat):
        s       = test_data[i : i+1].astype(np.float32)
        t0      = time.perf_counter()
        sess.run(None, {input_name: s})
        t1      = time.perf_counter()
        lat_times.append((t1 - t0) * 1000.0)  # ms
    lat_median_ms = float(np.median(lat_times))
    lat_mean_ms   = float(np.mean(lat_times))
    lat_p95_ms    = float(np.percentile(lat_times, 95))
    lat_p99_ms    = float(np.percentile(lat_times, 99))
    preds    = []
    t_start  = time.perf_counter()
    for start in range(0, n_total, batch_size):
        end   = min(start + batch_size, n_total)
        batch = test_data[
            start:end
        ].astype(np.float32)
        out   = sess.run(None, {input_name: batch})
        preds.append(out[0])
    t_end    = time.perf_counter()
    t_total  = t_end - t_start
    tput     = n_total / t_total  
    y_prob = np.concatenate(preds, axis=0)
    y_true = np.argmax(test_labels, axis=1)
    y_pred = np.argmax(y_prob,      axis=1)
    present_classes = np.union1d(
        np.unique(y_true),
        np.unique(y_pred))
    n_classes    = len(present_classes)
    target_names = [
        f'Gesture_{c+1}' for c in present_classes
    ]
    acc    = float(accuracy_score(y_true, y_pred))
    prec_w = float(precision_score(
        y_true, y_pred, average='weighted',
        zero_division=0, labels=present_classes))
    rec_w  = float(recall_score(
        y_true, y_pred, average='weighted',
        zero_division=0, labels=present_classes))
    f1_w   = float(f1_score(
        y_true, y_pred, average='weighted',
        zero_division=0, labels=present_classes))
    prec_m = float(precision_score(
        y_true, y_pred, average='macro',
        zero_division=0, labels=present_classes))
    rec_m  = float(recall_score(
        y_true, y_pred, average='macro',
        zero_division=0, labels=present_classes))
    f1_m   = float(f1_score(
        y_true, y_pred, average='macro',
        zero_division=0, labels=present_classes))
    kappa  = float(cohen_kappa_score(y_true, y_pred))
    report = classification_report(
        y_true, y_pred,
        labels       = present_classes,
        target_names = target_names,
        digits       = 4,
        zero_division= 0
    )
    cm = confusion_matrix(
        y_true, y_pred,
        labels=present_classes
    ).tolist()
    size_mb = os.path.getsize(onnx_path) / (1024**2)
    print(f"      [{label_name}] "
          f"Acc={acc:.4f} "
          f"F1w={f1_w:.4f} "
          f"κ={kappa:.4f} "
          f"Lat(med)={lat_median_ms:.2f}ms "
          f"Lat(p95)={lat_p95_ms:.2f}ms "
          f"Tput={tput:.1f}s/s "
          f"Size={size_mb:.4f}MB")

    return {
        "accuracy"                         : round(acc,           6),
        "precision_weighted"               : round(prec_w,        6),
        "recall_weighted"                  : round(rec_w,         6),
        "f1_weighted"                      : round(f1_w,          6),
        "precision_macro"                  : round(prec_m,        6),
        "recall_macro"                     : round(rec_m,         6),
        "f1_macro"                         : round(f1_m,          6),
        "kappa"                            : round(kappa,         6),
        "n_classes_detected"               : int(n_classes),
        "classification_report"            : report,
        "confusion_matrix"                 : cm,
        # Latency — single sample, CPU, 1 thread
        "latency_median_ms"                : round(lat_median_ms, 4),
        "latency_mean_ms"                  : round(lat_mean_ms,   4),
        "latency_p95_ms"                   : round(lat_p95_ms,    4),
        "latency_p99_ms"                   : round(lat_p99_ms,    4),
        "latency_samples_used"             : n_lat,
        "throughput_samples_per_sec"       : round(tput,          2),
        "throughput_batch_size"            : batch_size,
        "total_batched_inference_time_s"   : round(t_total,       4),
        "avg_inference_time_per_sample_ms" : round(lat_median_ms, 4),
        "model_size_mb"                    : round(size_mb,       4)
    }

def run_statistical_tests(all_seed_results,
                           model_name,
                           ts_name='all_rooms'):

    seeds = list(all_seed_results.keys())
    def get_seed_values(variant, metric='accuracy'):
        vals = []
        for s in seeds:
            v = (
                all_seed_results[s]
                .get(variant, {})
                .get(ts_name, {})
                .get(metric, None)
            )
            if v is not None:
                vals.append(float(v))
        return np.array(vals)

    variants_to_test = [
        'sta_int8',
        'sta_int16'
    ]

    metrics_to_test = [
        'accuracy',
        'f1_weighted',
        'f1_macro'
    ]
    comparisons = []
    fp32_vals = {}
    for m in metrics_to_test:
        fp32_vals[m] = get_seed_values('fp32', m)

    for vn in variants_to_test:
        for m in metrics_to_test:
            a = fp32_vals[m]
            b = get_seed_values(vn, m)
            if len(a) < 2 or len(b) < 2:
                continue
            if len(a) != len(b):
                continue
            diff = b - a
            # Wilcoxon signed-rank test
            try:
                if np.all(diff == 0):
                    w_stat = 0.0
                    w_p    = 1.0
                    w_note = 'identical'
                else:
                    w_result = wilcoxon(
                        diff,
                        zero_method = 'wilcox',
                        alternative = 'two-sided'
                    )
                    w_stat = float(w_result.statistic)
                    w_p    = float(w_result.pvalue)
                    w_note = (
                        'significant'
                        if w_p < 0.05
                        else 'not significant'
                    )
            except Exception as e:
                w_stat = None
                w_p    = None
                w_note = f'error: {e}'
            # Paired t-test
            try:
                t_result = ttest_rel(a, b)
                t_stat   = float(t_result.statistic)
                t_p      = float(t_result.pvalue)
                t_note   = (
                    'significant'
                    if t_p < 0.05
                    else 'not significant'
                )
            except Exception as e:
                t_stat = None
                t_p    = None
                t_note = f'error: {e}'
            comparisons.append({
                'comparison'   : f'{vn} vs fp32',
                'metric'       : m,
                'baseline_mean': float(np.mean(a)),
                'variant_mean' : float(np.mean(b)),
                'mean_diff'    : float(np.mean(diff)),
                'mean_diff_pct': float(
                    np.mean(diff) * 100
                ),
                'wilcoxon_stat': w_stat,
                'wilcoxon_p'   : w_p,
                'wilcoxon_note': w_note,
                't_stat'       : t_stat,
                't_p'          : t_p,
                't_note'       : t_note,
                'seed_values_baseline': a.tolist(),
                'seed_values_variant' : b.tolist(),
            })
    int8_int16_pairs = [('sta_int8', 'sta_int16')]
    for vn_a, vn_b in int8_int16_pairs:
        for m in metrics_to_test:
            a = get_seed_values(vn_a, m)
            b = get_seed_values(vn_b, m)
            if len(a) < 2 or len(b) < 2:
                continue
            if len(a) != len(b):
                continue
            diff = b - a
            try:
                if np.all(diff == 0):
                    w_stat = 0.0
                    w_p    = 1.0
                    w_note = 'identical'
                else:
                    w_result = wilcoxon(
                        diff,
                        zero_method = 'wilcox',
                        alternative = 'two-sided'
                    )
                    w_stat = float(w_result.statistic)
                    w_p    = float(w_result.pvalue)
                    w_note = (
                        'significant'
                        if w_p < 0.05
                        else 'not significant'
                    )
            except Exception as e:
                w_stat = None
                w_p    = None
                w_note = f'error: {e}'
            try:
                t_result = ttest_rel(a, b)
                t_stat   = float(t_result.statistic)
                t_p      = float(t_result.pvalue)
                t_note   = (
                    'significant'
                    if t_p < 0.05
                    else 'not significant'
                )
            except Exception as e:
                t_stat = None
                t_p    = None
                t_note = f'error: {e}'
            diff = b - a
            comparisons.append({
                'comparison'   : f'{vn_b} vs {vn_a}',
                'metric'       : m,
                'baseline_mean': float(np.mean(a)),
                'variant_mean' : float(np.mean(b)),
                'mean_diff'    : float(np.mean(diff)),
                'mean_diff_pct': float(
                    np.mean(diff) * 100
                ),
                'wilcoxon_stat': w_stat,
                'wilcoxon_p'   : w_p,
                'wilcoxon_note': w_note,
                't_stat'       : t_stat,
                't_p'          : t_p,
                't_note'       : t_note,
                'seed_values_baseline': a.tolist(),
                'seed_values_variant' : b.tolist(),
            })

    SEP = '─' * 90
    print(f"\n{'═'*90}")
    print(f"  Statistical Significance Tests")
    print(f"  Model    : {model_name}")
    print(f"  Test Set : {ts_name}")
    print(f"  N seeds  : {len(seeds)}")
    print(f"  Tests    : Wilcoxon signed-rank "
          f"(primary) + Paired t-test (secondary)")
    print(f"{'═'*90}")

    print(f"\n  {'Comparison':<26}"
          f"{'Metric':<16}"
          f"{'Mean':>10}"
          f"{'Wilcoxon p':>12}"
          f"{'t-test p':>10}"
          f"{'Significant?':>14}")
    print(f"  {SEP}")

    for c in comparisons:
        sig = (
            'YES'
            if (c['wilcoxon_p'] is not None
                and c['wilcoxon_p'] < 0.05)
            else '— no')
        w_p_str = (
            f"{c['wilcoxon_p']:.4f}"
            if c['wilcoxon_p'] is not None
            else 'N/A')
        t_p_str = (
            f"{c['t_p']:.4f}"
            if c['t_p'] is not None
            else 'N/A')
        diff_str = f"{c['mean_diff_pct']:+.4f}%"
        print(f"  {c['comparison']:<26}"
              f"{c['metric']:<16}"
              f"{diff_str:>10}"
              f"{w_p_str:>12}"
              f"{t_p_str:>10}"
              f"{sig:>14}")
    all_p = [
        c['wilcoxon_p']
        for c in comparisons
        if c['wilcoxon_p'] is not None
    ]
    n_sig = sum(p < 0.05 for p in all_p)
    print(f"\n  {'─'*90}")
    print(f"  Summary: {n_sig}/{len(all_p)} "
          f"comparisons are statistically significant "
          f"(Wilcoxon p < 0.05)")

    return comparisons

# PTQ 
def ptq_one_model(keras_model,
                  model_tag,
                  calibration_data,
                  calibration_labels,
                  test_sets_dict,
                  calib_seed):

    paths = {
        'fp32'     : os.path.join(
            PTQ_ONNX_DIR,
            f'{model_tag}_fp32.onnx'),
        'sta_int8' : os.path.join(
            PTQ_ONNX_DIR,
            f'{model_tag}_sta_int8.onnx'),
        'sta_int16': os.path.join(
            PTQ_ONNX_DIR,
            f'{model_tag}_sta_int16.onnx'),
    }

    # Step 1: FP32 export
    fp32_size = export_to_onnx(
        keras_model = keras_model,
        save_path   = paths['fp32'],
        opset       = OPSET
    )

    # Step 2: Quantize
    sizes = {'fp32': fp32_size}
    sizes['sta_int8'] = run_static_ptq(
        fp32_path        = paths['fp32'],
        save_path        = paths['sta_int8'],
        calibration_data = calibration_data,
        calibration_labels = calibration_labels, 
        quant_type       = 'int8',
        calib_seed       = calib_seed,
        n_calib_samples    = N_CALIB_SAMPLES 
    )
    sizes['sta_int16'] = run_static_ptq(
        fp32_path        = paths['fp32'],
        save_path        = paths['sta_int16'],
        calibration_data = calibration_data,
        calibration_labels = calibration_labels, 
        quant_type       = 'int16',
        calib_seed       = calib_seed,
        n_calib_samples    = N_CALIB_SAMPLES 
    )
    print(f"\n  Size Summary ({model_tag}):")
    print(f"  {'─'*50}")
    print(f"  {'Variant':<16}"
          f"{'Size(MB)':>10}"
          f"{'Reduction':>12}"
          f"  Note")
    print(f"  {'─'*50}")
    notes = {
        'fp32'     : 'baseline',
        'sta_int8' : 'w+a=INT8 Percentile calib',
        'sta_int16': 'w=INT8 a=INT16 Percentile calib',
    }
    for vname, vsize in sizes.items():
        if vname == 'fp32':
            red_str = '—'
        else:
            red     = (1 - vsize / fp32_size) * 100
            red_str = f"{red:.1f}%"
        print(f"  {vname:<16}"
              f"{vsize:>10.4f}"
              f"{red_str:>12}"
              f"  {notes[vname]}")
    results = {}
    for vname, vpath in paths.items():
        results[vname] = {}
        print(f"\n  [{vname}]")
        for ts_name, (ts_data, ts_labels) in \
                test_sets_dict.items():
            results[vname][ts_name] = evaluate_onnx(
                onnx_path   = vpath,
                test_data   = ts_data,
                test_labels = ts_labels,
                label_name  = f"{vname}|{ts_name}",
                n_warmup    = 5
            )
    return results

# AGGREGATION
def aggregate_ptq_across_seeds(all_seed_results):
    seeds         = list(all_seed_results.keys())
    variant_names = list(all_seed_results[seeds[0]].keys())
    test_set_keys = list(all_seed_results[seeds[0]][variant_names[0]].keys())

    scalar_metrics = [
        'accuracy',
        'precision_weighted',
        'recall_weighted',
        'f1_weighted',
        'precision_macro',
        'recall_macro',
        'f1_macro',
        'kappa',
        'model_size_mb',
        'latency_median_ms',
        'latency_mean_ms',
        'latency_p95_ms',
        'latency_p99_ms',
        'avg_inference_time_per_sample_ms',
        'throughput_samples_per_sec',
        'total_batched_inference_time_s',  
    ]
    aggregated = {}
    for vn in variant_names:
        aggregated[vn] = {}
        for ts in test_set_keys:
            aggregated[vn][ts] = {}
            for m in scalar_metrics:
                values = []
                for s in seeds:
                    v = (
                        all_seed_results[s]
                        .get(vn, {})
                        .get(ts, {})
                        .get(m, None)
                    )
                    if v is not None and v != -1:
                        values.append(float(v))
                if values:
                    aggregated[vn][ts][m] = {
                        'mean'  : float(np.mean(values)),
                        'std'   : float(np.std(values)),
                        'min'   : float(np.min(values)),
                        'max'   : float(np.max(values)),
                        'values': values
                    }
    return aggregated

In [ ]:
# ONNX Runtime uses numpy directly.
SEEDS  = [42, 100, 200, 300, 500]
test_sets_dict = {
    'all_rooms': (data_test,   label_test),
    'room2'    : (data_test_1, label_test_1),
    'room3'    : (data_test_2, label_test_2),
}
# Train Loop for two models
all_ptq_m1 = {}
all_ptq_m2 = {}

for seed in SEEDS:
    print(f"  SEED {seed}  "    
          f"({SEEDS.index(seed)+1}/{len(SEEDS)})")
    train_s, _, label_s, _ = train_test_split(
        data_train, label_train,
        test_size    = 0.2,
        shuffle      = True,
        random_state = seed)
    rng       = np.random.RandomState(seed)
    pool_size = min(
        N_CALIB_SAMPLES * 3,
        len(train_s)
    )
    calib_idx = rng.choice(
        len(train_s),
        size    = min(N_CALIB_SAMPLES * 3, len(train_s)),
        replace = False
        )
    calib_data   = train_s[calib_idx]
    calib_labels = label_s[calib_idx]
    print(f"  Calibration pool : {pool_size} samples")
    print(f"  Target after strat: {N_CALIB_SAMPLES}")

    print(f"\n  [model_1] Loading seed {seed}...")
    m1_path = os.path.join(
        SAVE_DIR,
        f'model_1_seed{seed}.keras'
    )
    if not os.path.exists(m1_path):
        print(f"  [ERROR] Not found: {m1_path}")
        print(f"  [SKIP]  model_1 seed {seed}")
    else:
        model_1 = tf.keras.models.load_model(m1_path)
        print(f"  model_1 seed {seed} loaded")

        all_ptq_m1[seed] = ptq_one_model(
            keras_model      = model_1,
            model_tag        = f'model_1_seed{seed}',
            calibration_data = calib_data,
            calibration_labels = calib_labels,
            test_sets_dict   = test_sets_dict,
            calib_seed       = seed
        )
    print(f"\n  [model_2] Loading seed {seed}...")
    m2_path = os.path.join(
        SAVE_DIR,
        f'model_2_seed{seed}.weights.h5'
    )
    if not os.path.exists(m2_path):
        print(f"  [ERROR] Not found: {m2_path}")
        print(f"  [SKIP]  model_2 seed {seed}")

    else:
        model_2 = transformer_model(
            input_shape = (36, 20, 20, 1),
            n_class     = 6,
            num_layers  = 2
        )
        dummy = np.zeros((1, 36, 20, 20, 1),dtype=np.float32)
        _ = model_2(dummy, training=False)
        model_2.load_weights(m2_path)
        print(f" model_2 seed {seed} loaded")
        all_ptq_m2[seed] = ptq_one_model(
            keras_model      = model_2,
            model_tag        = f'model_2_seed{seed}',
            calibration_data = calib_data,
            calibration_labels = calib_labels,
            test_sets_dict   = test_sets_dict,
            calib_seed       = seed
        )

print("  Aggregating results")
for label, store in [('model_1', all_ptq_m1),
                     ('model_2', all_ptq_m2)]:
    missing = [s for s in SEEDS if s not in store]
    if missing:
        print(f"  [WARN] {label} missing: {missing}")
    else:
        print(f"  {label}: "
              f"all {len(SEEDS)} seeds present")

aggregated_m1 = aggregate_ptq_across_seeds(all_ptq_m1)
aggregated_m2 = aggregate_ptq_across_seeds(all_ptq_m2)
print("  STEP 4: Statistical significance tests")
stat_results_m1 = {}
stat_results_m2 = {}
for ts_name in ['all_rooms', 'room2', 'room3']:
    print(f"  Model 1 — CNN-RNN | {ts_name}")
    stat_results_m1[ts_name] = run_statistical_tests(
        all_seed_results = all_ptq_m1,
        model_name       = 'Model 1 — CNN-RNN',
        ts_name          = ts_name)
    print(f"  Model 2 — Transformer | {ts_name}")
    print(f"  {'─'*60}")
    stat_results_m2[ts_name] = run_statistical_tests(
        all_seed_results = all_ptq_m2,
        model_name       = 'Model 2 — Transformer',
        ts_name          = ts_name)

QAT onnx

In [ ]:
# Build Fake Quantization Layer
class FakeQuantize(tf.keras.layers.Layer):
    def __init__(self, num_bits=8,
                 symmetric=True,
                 per_channel=False,
                 channel_axis=-1,
                 ema_decay=0.9,
                 **kwargs):
        super().__init__(**kwargs)
        self.num_bits     = num_bits
        self.symmetric    = symmetric
        self.per_channel  = per_channel
        self.channel_axis = channel_axis
        self.ema_decay    = ema_decay
        if symmetric:
            self.q_min = -(2 ** (num_bits - 1))
            self.q_max =  (2 ** (num_bits - 1)) - 1
        else:
            self.q_min = 0
            self.q_max = (2 ** num_bits) - 1
    def build(self, input_shape):
        if self.per_channel:
            axis = (
                len(input_shape) +
                self.channel_axis
            ) % len(input_shape)
            n_channels = input_shape[axis]
            shape = (n_channels,)
        else:
            shape = (1,)
        self.ema_min = self.add_weight(
            name        = 'ema_min',
            shape       = shape,
            initializer = tf.zeros_initializer(),
            trainable   = False)
        self.ema_max = self.add_weight(
            name        = 'ema_max',
            shape       = shape,
            initializer = tf.ones_initializer(),
            trainable   = False)
        self.initialized = self.add_weight(
            name        = 'initialized',
            shape       = (),
            initializer = tf.zeros_initializer(),
            trainable   = False,
            dtype       = tf.float32)
        super().build(input_shape)

    def compute_output_shape(self, input_shape):
        return input_shape
    def _update_ema(self, x):
        if self.per_channel:
            axes = [
                i for i in range(len(x.shape))
                if i != (
                    len(x.shape) +
                    self.channel_axis
                ) % len(x.shape)
            ]
            x_min = tf.reduce_min(x, axis=axes)
            x_max = tf.reduce_max(x, axis=axes)
        else:
            x_min = tf.reshape(tf.reduce_min(x), (1,))
            x_max = tf.reshape(tf.reduce_max(x), (1,))
        new_min = tf.where(
            self.initialized < 0.5,
            x_min,
            self.ema_decay * self.ema_min +
            (1 - self.ema_decay) * x_min)
        new_max = tf.where(
            self.initialized < 0.5,
            x_max,
            self.ema_decay * self.ema_max +
            (1 - self.ema_decay) * x_max)
        self.ema_min.assign(new_min)
        self.ema_max.assign(new_max)
        self.initialized.assign(1.0)

    def _compute_scale_zp(self):
        if self.symmetric:
            abs_max = tf.maximum(
                tf.abs(self.ema_min),
                tf.abs(self.ema_max)
            )
            abs_max = tf.maximum(abs_max, 1e-8)
            scale   = abs_max / float(self.q_max)
            zp      = tf.zeros_like(scale)
        else:
            range_val = tf.maximum(self.ema_max - self.ema_min, 1e-8)
            scale = range_val / float(self.q_max - self.q_min)
            zp = tf.round(self.q_min - self.ema_min / scale)
            zp = tf.clip_by_value(zp,float(self.q_min),float(self.q_max))
        return scale, zp

    def call(self, x, training=False):
        if training:
            self._update_ema(x)
        scale, zp = self._compute_scale_zp()
        x_int = tf.round(x / scale + zp)
        x_int = tf.clip_by_value(x_int,float(self.q_min),float(self.q_max))
        x_dequant = (x_int - zp) * scale
        return x + tf.stop_gradient(x_dequant - x)

    def get_config(self):
        config = super().get_config()
        config.update({
            'num_bits'    : self.num_bits,
            'symmetric'   : self.symmetric,
            'per_channel' : self.per_channel,
            'channel_axis': self.channel_axis,
            'ema_decay'   : self.ema_decay
        })
        return config

# Build QAT Layers for Model 1 (CR)
# QAT wrapper for Conv2D
class QATConv2D(tf.keras.layers.Layer):
    def __init__(self, filters, kernel_size,
                 activation = None,
                 padding    = 'same',
                 num_bits   = 8,
                 **kwargs):
        super().__init__(**kwargs)
        self.filters     = filters
        self.kernel_size = kernel_size if isinstance(
            kernel_size, tuple
        ) else (kernel_size, kernel_size)
        self.activation  = activation
        self.padding     = padding.lower()
        self.num_bits    = num_bits

    def build(self, input_shape):
        in_channels = input_shape[-1]
        self.kernel = self.add_weight(
            name        = 'kernel',
            shape       = (self.kernel_size[0],
                           self.kernel_size[1],
                           in_channels,
                           self.filters),
            initializer = 'glorot_uniform',
            trainable   = True)
        self.bias = self.add_weight(
            name        = 'bias',
            shape       = (self.filters,),
            initializer = 'zeros',
            trainable   = True)
        self.fq_weight = FakeQuantize(
            num_bits     = self.num_bits,
            symmetric    = True,
            per_channel  = True,
            channel_axis = -1,
            name         = 'fq_weight')
        self.fq_act = FakeQuantize(
            num_bits    = self.num_bits,
            symmetric   = False,
            per_channel = False,
            name        = 'fq_act')
        if self.activation == 'relu':
            self.act_fn = tf.keras.layers.ReLU()
        elif self.activation == 'gelu':
            self.act_fn = tf.keras.layers.Activation(
                'gelu')
        else:
            self.act_fn = None
        super().build(input_shape)

    def compute_output_shape(self, input_shape):
        input_shape = tf.TensorShape(input_shape)
        h_in = input_shape[1]
        w_in = input_shape[2]
        if self.padding == 'same':
            h_out = h_in
            w_out = w_in
        else:  
            h_out = h_in - self.kernel_size[0] + 1
            w_out = w_in - self.kernel_size[1] + 1
        return tf.TensorShape(
            [input_shape[0],
             h_out,
             w_out,
             self.filters])
    
    def call(self, x, training=False):
        w_fq = self.fq_weight(self.kernel, training=training)
        padding_str = self.padding.upper()
        out = tf.nn.conv2d(
            input   = x,
            filters = w_fq,
            strides = [1, 1, 1, 1],
            padding = padding_str)
        out = tf.nn.bias_add(out, self.bias)
        if self.act_fn is not None:
            out = self.act_fn(out)
        out = self.fq_act(out, training=training)
        return out

    def get_config(self):
        config = super().get_config()
        config.update({
            'filters'    : self.filters,
            'kernel_size': self.kernel_size,
            'activation' : self.activation,
            'padding'    : self.padding,
            'num_bits'   : self.num_bits
        })
        return config

class QATDense(tf.keras.layers.Layer):
    def __init__(self,
                 units,
                 activation = None,
                 use_bias   = True,
                 num_bits   = 8,
                 **kwargs):
        super().__init__(**kwargs)
        self.units      = units
        self.activation = activation
        self.use_bias   = use_bias
        self.num_bits   = num_bits
    def build(self, input_shape):
        in_dim = int(input_shape[-1])
        self.kernel = self.add_weight(
            name        = 'kernel',
            shape       = (in_dim, self.units),
            initializer = 'glorot_uniform',
            trainable   = True)
        if self.use_bias:
            self.bias = self.add_weight(
                name        = 'bias',
                shape       = (self.units,),
                initializer = 'zeros',
                trainable   = True)
        else:
            self.bias = None

        #  FakeQuantize for weights 
        self.fq_weight = FakeQuantize(
            num_bits     = self.num_bits,
            symmetric    = True,
            per_channel  = True,
            channel_axis = -1,
            name         = 'fq_weight')

        # FakeQuantize for activations 
        self.fq_act = FakeQuantize(
            num_bits    = self.num_bits,
            symmetric   = False,
            per_channel = False,
            name        = 'fq_act')
        if self.activation is not None:
            self.act_fn = tf.keras.layers.Activation(
                self.activation,
                name = f'act_{self.activation}')
        else:
            self.act_fn = None
        super().build(input_shape)


    def compute_output_shape(self, input_shape):
        input_shape = tf.TensorShape(input_shape)
        return tf.TensorShape(input_shape[:-1].as_list() + [self.units])

    def call(self, x, training=False):
        w_fq = self.fq_weight(self.kernel, training=training)
        out = tf.matmul(x, w_fq)
        if self.bias is not None:
            out = tf.nn.bias_add(out, self.bias)
        if self.act_fn is not None:
            out = self.act_fn(out)
        out = self.fq_act(out, training=training)
        return out

    def get_config(self):
        config = super().get_config()
        config.update({
            'units'     : self.units,
            'activation': self.activation,
            'use_bias'  : self.use_bias,
            'num_bits'  : self.num_bits,
        })
        return config
    @classmethod
    def from_config(cls, config):
        return cls(**config)


# QAT version of Model 1 (CNN-RNN)
def build_qat_model_1(input_shape, n_class, num_bits=8):
    model_input = Input( shape = input_shape, dtype = 'float32',name  = 'model_input')
    x = TimeDistributed(QATConv2D(filters=32,kernel_size=(5, 5),activation='relu',padding='valid', num_bits= num_bits,name = 'qat_conv1'), name = 'td_conv1')(model_input)
    x = TimeDistributed( MaxPooling2D(pool_size=(2, 2), name='pool1'),name = 'td_pool1')(x)
    x = TimeDistributed(Flatten(name='flatten'),name = 'td_flatten')(x)
    x = TimeDistributed(QATDense(units = 256, activation = 'gelu', use_bias   = True, num_bits = num_bits, name = 'qat_dense_proj'), name = 'td_dense_proj')(x)
    x = Bidirectional(GRU(128, return_sequences = True, name = 'gru'), name = 'bidirectional_gru')(x)
    x = Dropout(0.2)(x)
    x = LSTM(128, return_sequences = True, name = 'lstm')(x)
    x = Dropout(0.2)(x)
    x = Flatten()(x)
    x = FakeQuantize(
        num_bits    = num_bits,
        symmetric   = False,
        per_channel = False,
        name        = 'fq_pre_output')(x)
    model_output = QATDense(
        units      = n_class,
        activation = 'softmax',
        use_bias   = True,
        num_bits   = num_bits,
        name       = 'qat_dense_output')(x)
    model = Model(
        inputs  = model_input,
        outputs = model_output,
        name    = f'qat_model_1_int{num_bits}')
    model.compile(
        optimizer = tf.keras.optimizers.Adam(lr=1e-4),
        loss      = CategoricalCrossentropy(label_smoothing=0.1),
        metrics   = ['accuracy'])
    return model



# Model 2
# Transformer model
class QATScaledDotProductAttention(tf.keras.layers.Layer):
    def __init__(self, dropout_ratio=0.1,
                 num_bits=8, **kwargs):
        super().__init__(**kwargs)
        self.dropout_ratio = dropout_ratio
        self.num_bits      = num_bits
        self.dropout       = Dropout(dropout_ratio)
    def build(self, input_shape):
        self.fq_q = FakeQuantize(
            num_bits    = self.num_bits,
            symmetric   = False,
            per_channel = False,
            name        = 'fq_q')
        self.fq_k = FakeQuantize(
            num_bits    = self.num_bits,
            symmetric   = False,
            per_channel = False,
            name        = 'fq_k')
        self.fq_v = FakeQuantize(
            num_bits    = self.num_bits,
            symmetric   = False,
            per_channel = False,
            name        = 'fq_v')
        self.fq_out = FakeQuantize(
            num_bits    = self.num_bits,
            symmetric   = False,
            per_channel = False,
            name        = 'fq_attn_out')
        super().build(input_shape)

    def call(self, q, k, v,
             mask=None, training=False):
        q = self.fq_q(q, training=training)
        k = self.fq_k(k, training=training)
        v = self.fq_v(v, training=training)
        matmul_qk = tf.matmul(q, k, transpose_b=True)
        dk  = tf.cast(tf.shape(k)[-1], tf.float32)
        scaled = matmul_qk / tf.math.sqrt(dk)
        if mask is not None:
            scaled += (mask * -1e9)
        attn_weights = tf.nn.softmax(scaled, axis=-1)
        attn_weights = self.dropout(attn_weights, training=training)
        output = tf.matmul(attn_weights, v)
        output = self.fq_out(output, training=training)
        return output
    def get_config(self):
        config = super().get_config()
        config.update({
            'dropout_ratio': self.dropout_ratio,
            'num_bits'     : self.num_bits
        })
        return config

# Multi-head
class QATMultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads,
                 dropout_ratio=0.1,
                 num_bits=8, **kwargs):
        super().__init__(**kwargs)
        self.num_heads     = num_heads
        self.d_model       = d_model
        self.dropout_ratio = dropout_ratio
        self.num_bits      = num_bits
        assert d_model % num_heads == 0
        self.depth = d_model // num_heads
    def build(self, input_shape):
        self.wq    = Dense(self.d_model)
        self.wk    = Dense(self.d_model)
        self.wv    = Dense(self.d_model)
        self.dense = Dense(self.d_model)
        self.fq_q_proj = FakeQuantize(
            num_bits    = self.num_bits,
            symmetric   = False,
            per_channel = False,
            name        = 'fq_q_proj')
        self.fq_k_proj = FakeQuantize(
            num_bits    = self.num_bits,
            symmetric   = False,
            per_channel = False,
            name        = 'fq_k_proj')
        self.fq_v_proj = FakeQuantize(
            num_bits    = self.num_bits,
            symmetric   = False,
            per_channel = False,
            name        = 'fq_v_proj')
        self.fq_out = FakeQuantize(
            num_bits    = self.num_bits,
            symmetric   = False,
            per_channel = False,
            name        = 'fq_mha_out')
        self.attention = QATScaledDotProductAttention(
            dropout_ratio = self.dropout_ratio,
            num_bits      = self.num_bits,
            name          = 'qat_sdpa')
        self.dropout = Dropout(self.dropout_ratio)
        super().build(input_shape)

    def split_heads(self, x, batch_size):
        x = tf.reshape(x,(batch_size, -1,self.num_heads, self.depth))
        return tf.transpose(x, perm=[0,2,1,3])
    def call(self, q, k, v,
             mask=None, training=False):
        batch_size = tf.shape(q)[0]
        q = self.fq_q_proj(self.wq(q), training=training)
        k = self.fq_k_proj(self.wk(k), training=training)
        v = self.fq_v_proj(self.wv(v), training=training)
        q = self.split_heads(q, batch_size)
        k = self.split_heads(k, batch_size)
        v = self.split_heads(v, batch_size)
        attn = self.attention(q, k, v, mask, training)
        attn = tf.transpose(attn, perm=[0,2,1,3])
        concat = tf.reshape(attn, (batch_size, -1, self.d_model))
        output = self.fq_out(self.dense(concat), training=training)
        return output
    def get_config(self):
        config = super().get_config()
        config.update({
            'd_model'      : self.d_model,
            'num_heads'    : self.num_heads,
            'dropout_ratio': self.dropout_ratio,
            'num_bits'     : self.num_bits
        })
        return config

# Encoder Layer
class QATTransformerEncoderLayer(tf.keras.layers.Layer):
    def __init__(self, ff_dim, dropout_ratio,
                 num_bits=8, **kwargs):
        super().__init__(**kwargs)
        self.ff_dim        = ff_dim
        self.dropout_ratio = dropout_ratio
        self.num_bits      = num_bits

    def build(self, input_shape):
        d_model = input_shape[-1]
        self.attention   = QATMultiHeadAttention(
            d_model       = d_model,
            num_heads     = 4,
            dropout_ratio = self.dropout_ratio,
            num_bits      = self.num_bits,
            name          = 'qat_mha')
        self.layer_norm1 = LayerNormalization(epsilon=1e-6)
        self.dropout1    = Dropout(self.dropout_ratio)
        self.dense1      = Dense(self.ff_dim, activation = 'gelu', kernel_regularizer  = l2(0.01))
        self.dense2      = Dense(d_model, kernel_regularizer  = l2(0.01))
        self.layer_norm2 = LayerNormalization(epsilon=1e-6)
        self.dropout2    = Dropout(self.dropout_ratio)
        self.fq_ffn_mid = FakeQuantize(
            num_bits    = self.num_bits,
            symmetric   = False,
            per_channel = False,
            name        = 'fq_ffn_mid')
        self.fq_ffn_out = FakeQuantize(
            num_bits    = self.num_bits,
            symmetric   = False,
            per_channel = False,
            name        = 'fq_ffn_out')
        super().build(input_shape)

    def call(self, inputs, training=False):
        seq_len = inputs.shape[1]
        d_model = inputs.shape[2]
        pos_enc = positional_encoding(seq_len, d_model)
        x = inputs + pos_enc
        attn_out = self.attention(x, x, x, training=training)
        attn_out = self.dropout1(attn_out, training=training)
        out1 = self.layer_norm1(inputs + attn_out)
        ffn = self.fq_ffn_mid(self.dense1(out1), training=training)
        ffn = self.fq_ffn_out(self.dense2(ffn), training=training)
        ffn = self.dropout2(ffn, training=training)
        return self.layer_norm2(out1 + ffn)

    def get_config(self):
        config = super().get_config()
        config.update({
            'ff_dim'       : self.ff_dim,
            'dropout_ratio': self.dropout_ratio,
            'num_bits'     : self.num_bits
        })
        return config

# Model_2
def build_qat_model_2(input_shape,
                      n_class,
                      num_layers = 2,
                      num_bits   = 8):
    model_input = Input(shape = input_shape, dtype = 'float32', name  = 'model_input' )
    x = TimeDistributed(Flatten())(model_input)
    x = TimeDistributed(Dense(128, activation='gelu'))(x)
    x = TimeDistributed(Dropout(0.5))(x)
    for i in range(num_layers):
        x = QATTransformerEncoderLayer(
            ff_dim        = 256,
            dropout_ratio = 0.1,
            num_bits      = num_bits,
            name          = f'qat_encoder_{i}')(x)
    x = Flatten()(x)
    x = FakeQuantize(
        num_bits    = num_bits,
        symmetric   = False,
        per_channel = False,
        name        = 'fq_pre_output')(x)
    model_output = Dense(
        n_class,
        activation = 'softmax',
        name       = 'name_model_output')(x)
    model = Model(inputs  = model_input, outputs = model_output)
    model.compile(
        optimizer = tf.keras.optimizers.Adam(learning_rate = 1e-4),
        loss    = CategoricalCrossentropy(label_smoothing=0.1),
        metrics = ['accuracy'])
    print(f"\n[✓] QAT Model 2 (INT{num_bits}) built")
    print(f"    Layers  : {num_layers} transformers")
    print(f"    Num bits: {num_bits}")
    model.summary()
    return model

In [ ]:
# Transfer weights
def transfer_fp32_to_qat(fp32_model, qat_model):
    fp32_conv   = []
    fp32_dense  = []
    fp32_rnn    = []
    for layer in fp32_model.layers:
        if (isinstance(layer, TimeDistributed) and
                isinstance(layer.layer, Conv2D)):
            fp32_conv.append(layer.layer)
        elif (isinstance(layer, TimeDistributed) and
                isinstance(layer.layer, Dense)):
            fp32_dense.append(layer.layer)
        elif isinstance(layer, (Bidirectional,
                                LSTM, GRU)):
            fp32_rnn.append(layer)
        elif isinstance(layer, Dense):
            fp32_dense.append(layer)
    qat_conv   = []
    qat_dense  = []
    qat_rnn    = []
    for layer in qat_model.layers:
        if (isinstance(layer, TimeDistributed) and
                isinstance(layer.layer, QATConv2D)):
            qat_conv.append(layer.layer)
        elif (isinstance(layer, TimeDistributed) and
                isinstance(layer.layer, QATDense)):
            qat_dense.append(layer.layer)
        elif isinstance(layer, (Bidirectional,
                                LSTM, GRU)):
            qat_rnn.append(layer)
        elif isinstance(layer, QATDense):
            qat_dense.append(layer)
    transferred = 0
    for fp32_c, qat_c in zip(fp32_conv, qat_conv):
        fp32_w = fp32_c.get_weights()  
        qat_w  = qat_c.get_weights()  
        qat_w[0] = fp32_w[0]  
        qat_w[1] = fp32_w[1]  
        qat_c.set_weights(qat_w)
        print(f" Conv2D → QATConv2D: "
              f"{fp32_w[0].shape}")
        transferred += 1
    for fp32_d, qat_d in zip(fp32_dense, qat_dense):
        fp32_w = fp32_d.get_weights()  
        qat_w  = qat_d.get_weights()   
        qat_w[0] = fp32_w[0]  
        qat_w[1] = fp32_w[1]  
        qat_d.set_weights(qat_w)
        print(f"    Dense → QATDense: "
              f"{fp32_w[0].shape}")
        transferred += 1
    for fp32_r, qat_r in zip(fp32_rnn, qat_rnn):
        try:
            qat_r.set_weights(fp32_r.get_weights())
            print(f"   RNN copy: {fp32_r.name} "
                  f"→ {qat_r.name}")
            transferred += 1
        except Exception as e:
            print(f"    RNN copy failed "
                  f"{fp32_r.name}: {e}")

    print(f"\n  [Transfer] Done: "
          f"{transferred} layers transferred")
    return transferred


def transfer_fp32_to_qat_m2(fp32_model, qat_model):
    transferred = 0
    fp32_named = []
    for layer in fp32_model.layers:
        ws = layer.get_weights()
        if ws:
            shapes = [w.shape for w in ws]
            print(f"    {layer.name:<50} {shapes}")
            fp32_named.append(layer)
    qat_named = []
    for layer in qat_model.layers:
        ws = layer.get_weights()
        if ws:
            shapes = [w.shape for w in ws]
            print(f"    {layer.name:<50} {shapes}")
            qat_named.append(layer)
    fp32_td_dense = []
    for layer in fp32_model.layers:
        if (isinstance(layer, TimeDistributed) and
                isinstance(layer.layer, Dense) and
                layer.layer.units == 128):
            fp32_td_dense.append(layer)
            print(f"    Found FP32 TD Dense: {layer.name} "
                  f"units={layer.layer.units}")
    qat_td_dense = []
    for layer in qat_model.layers:
        if (isinstance(layer, TimeDistributed) and
                isinstance(layer.layer, Dense) and
                layer.layer.units == 128):
            qat_td_dense.append(layer)
            print(f"    Found QAT TD Dense: {layer.name} "
                  f"units={layer.layer.units}")

    for fp32_l, qat_l in zip(fp32_td_dense, qat_td_dense):
        fp32_w = fp32_l.get_weights()  
        qat_w  = qat_l.get_weights()   
        if fp32_w[0].shape == qat_w[0].shape:
            qat_l.set_weights(fp32_w)
            print(f"    [✓] TD Dense: {fp32_l.name} → "
                  f"{qat_l.name} "
                  f"kernel={fp32_w[0].shape}")
            transferred += 1
        else:
            print(f" Shape mismatch: "
                  f"{fp32_w[0].shape} vs {qat_w[0].shape}")
            
    fp32_encoders = []
    for layer in fp32_model.layers:
        if isinstance( layer, TransformerMultiHeadAttentionEncoderLayer):
            fp32_encoders.append(layer)
    qat_encoders = []
    for layer in qat_model.layers:
        if isinstance(layer, QATTransformerEncoderLayer):
            qat_encoders.append(layer)
    if len(fp32_encoders) != len(qat_encoders):
        print(f"  Encoder count mismatch: "
              f"{len(fp32_encoders)} FP32 vs "
              f"{len(qat_encoders)} QAT")
    else:
        print(f" Encoder count matches: "
              f"{len(fp32_encoders)}")
    for enc_idx, (fp32_enc, qat_enc) in enumerate(zip(fp32_encoders, qat_encoders)):
        print(f"\n    Transferring encoder {enc_idx}: "
              f"{fp32_enc.name} → {qat_enc.name}")
        enc_transferred = _transfer_encoder_weights(fp32_enc, qat_enc, enc_idx)
        transferred += enc_transferred
    fp32_output_dense = []
    for layer in fp32_model.layers:
        if (isinstance(layer, Dense) and
                not isinstance(layer, TimeDistributed) and
                layer.units == 6):
            fp32_output_dense.append(layer)
            print(f"    Found FP32 output Dense: "
                  f"{layer.name} units={layer.units}")
    qat_output_dense = []
    for layer in qat_model.layers:
        if (isinstance(layer, Dense) and
                not isinstance(layer, TimeDistributed) and
                layer.units == 6):
            qat_output_dense.append(layer)
            print(f"    Found QAT output Dense: "
                  f"{layer.name} units={layer.units}")
    for fp32_l, qat_l in zip(fp32_output_dense, qat_output_dense):
        fp32_w = fp32_l.get_weights()
        qat_w  = qat_l.get_weights()
        if fp32_w[0].shape == qat_w[0].shape:
            qat_l.set_weights(fp32_w)
            print(f"    [✓] Output Dense: {fp32_l.name} → "
                  f"{qat_l.name} "
                  f"kernel={fp32_w[0].shape}")
            transferred += 1
        else:
            print(f"    [✗] Shape mismatch: "
                  f"{fp32_w[0].shape} vs {qat_w[0].shape}")

    print(f"\n  [Transfer M2] Done: "
          f"{transferred} layers transferred")
    return transferred


def _transfer_encoder_weights(fp32_enc, qat_enc, enc_idx):

    transferred = 0
    mha_sublayers = [
        ('wq',    'wq'),
        ('wk',    'wk'),
        ('wv',    'wv'),
        ('dense', 'dense'),]
    for fp32_attr, qat_attr in mha_sublayers:
        try:
            fp32_sub = getattr(fp32_enc.attention, fp32_attr)
            qat_sub  = getattr(qat_enc.attention,  qat_attr)
            fp32_w = fp32_sub.get_weights()
            qat_w  = qat_sub.get_weights()
            if not fp32_w:
                print(f" {fp32_attr} has no weights, "
                      f"skipping")
                continue
            if fp32_w[0].shape == qat_w[0].shape:
                qat_sub.set_weights(fp32_w)
                print(f"      [✓] MHA.{fp32_attr}: "
                      f"kernel={fp32_w[0].shape}")
                transferred += 1
            else:
                print(f"      [✗] MHA.{fp32_attr} shape "
                      f"mismatch: "
                      f"{fp32_w[0].shape} vs "
                      f"{qat_w[0].shape}")

        except AttributeError as e:
            print(f"      [!] Attribute error for "
                  f"{fp32_attr}: {e}")
    try:
        fp32_w = fp32_enc.layer_norm1.get_weights()
        qat_w  = qat_enc.layer_norm1.get_weights()
        if fp32_w and fp32_w[0].shape == qat_w[0].shape:
            qat_enc.layer_norm1.set_weights(fp32_w)
            print(f"  LayerNorm1: "
                  f"gamma={fp32_w[0].shape}")
            transferred += 1
    except Exception as e:
        print(f"  LayerNorm1 failed: {e}")
    try:
        fp32_w = fp32_enc.dense1.get_weights()
        qat_w  = qat_enc.dense1.get_weights()
        if fp32_w and fp32_w[0].shape == qat_w[0].shape:
            qat_enc.dense1.set_weights(fp32_w)
            print(f" FFN Dense1: "
                  f"kernel={fp32_w[0].shape}")
            transferred += 1
    except Exception as e:
        print(f"  FFN Dense1 failed: {e}")
    try:
        fp32_w = fp32_enc.dense2.get_weights()
        qat_w  = qat_enc.dense2.get_weights()
        if fp32_w and fp32_w[0].shape == qat_w[0].shape:
            qat_enc.dense2.set_weights(fp32_w)
            print(f"  FFN Dense2: "
                  f"kernel={fp32_w[0].shape}")
            transferred += 1
    except Exception as e:
        print(f"  FFN Dense2 failed: {e}")
    try:
        fp32_w = fp32_enc.layer_norm2.get_weights()
        qat_w  = qat_enc.layer_norm2.get_weights()
        if fp32_w and fp32_w[0].shape == qat_w[0].shape:
            qat_enc.layer_norm2.set_weights(fp32_w)
            print(f"  LayerNorm2: "
                  f"gamma={fp32_w[0].shape}")
            transferred += 1
    except Exception as e:
        print(f"  LayerNorm2 failed: {e}")

    print(f"      Encoder {enc_idx}: "
          f"{transferred} sublayers transferred")
    return transferred

In [ ]:
# Prepare QAT model
def train_qat_model(qat_model,
                          fp32_model,
                          model_tag,
                          train_data,
                          train_labels,
                          val_data,
                          val_labels,
                          seed,
                          epochs=50,
                          batch_size=32):
    has_transformer = any(
        isinstance(layer, QATTransformerEncoderLayer)
        for layer in qat_model.layers)
    has_conv = any(
        isinstance(layer, TimeDistributed) and
        isinstance(layer.layer, QATConv2D)
        for layer in qat_model.layers)
    if has_transformer:
        print(f"  [QAT] Detected Model 2 "
              f"(Transformer) — using M2 transfer")
        transferred = transfer_fp32_to_qat_m2(fp32_model, qat_model)
    elif has_conv:
        print(f"  [QAT] Detected Model 1 "
              f"(CNN-RNN) — using M1 transfer")
        transferred = transfer_fp32_to_qat(fp32_model, qat_model)
    else:
        raise RuntimeError("Cannot detect model type. ")
    if transferred == 0:
        raise RuntimeError( "Weight transfer failed — 0 layers transferred. ")

    print(f"[QAT] {transferred} layers transferred successfully")

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor              = 'val_accuracy',
            patience             = 10,
            restore_best_weights = True,
            verbose              = 1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor  = 'val_loss',
            factor   = 0.5,
            patience = 5,
            min_lr   = 1e-6,
            verbose  = 1)]
    print(f"  [QAT] Fine-tuning {model_tag}...")
    history = qat_model.fit(
        train_data, train_labels,
        validation_data = (val_data, val_labels),
        epochs          = epochs,
        batch_size      = batch_size,
        callbacks       = callbacks,
        verbose         = 1)

    save_path = os.path.join(QAT_SAVE_DIR, f'{model_tag}_qat.weights.h5')
    qat_model.save_weights(save_path)
    print(f" QAT weights saved: {save_path}")
    return history


# QAT model
def qat_one_model(fp32_model,
                  model_tag,
                  model_builder_fn,
                  build_kwargs,
                  train_data,
                  train_labels,
                  val_data,
                  val_labels,
                  calibration_data,
                  calibration_labels,
                  test_sets_dict,
                  calib_seed,
                  epochs     = n_epochs,
                  batch_size = 32):

    paths = {
        'fp32_onnx_reference': os.path.join(
            QAT_ONNX_DIR,
            f'{model_tag}_fp32.onnx'),
        'qat_int8': {
            'fp32_onnx' : os.path.join(
                QAT_ONNX_DIR,
                f'{model_tag}_qat_int8_fp32.onnx'),
            'quant_onnx': os.path.join(
                QAT_ONNX_DIR,
                f'{model_tag}_qat_int8.onnx'),},
        'qat_int16': {
            'fp32_onnx' : os.path.join(
                QAT_ONNX_DIR,
                f'{model_tag}_qat_int16_fp32.onnx'),
            'quant_onnx': os.path.join(
                QAT_ONNX_DIR,
                f'{model_tag}_qat_int16.onnx'),},}

    bit_map = {'qat_int8' : 8,'qat_int16': 16}
    quant_type_map = {'qat_int8' : 'int8','qat_int16': 'int16'}
    sizes   = {}
    results = {}
    fp32_ref_size = export_to_onnx( 
        keras_model = fp32_model,
        save_path   = paths['fp32_onnx_reference'],
        opset       = OPSET)
    sizes['fp32_reference'] = fp32_ref_size
    print(f"  FP32 ONNX reference: "
          f"{fp32_ref_size:.4f} MB  "
          f"(this is your ~4.024 MB baseline)")
    for vname, num_bits in bit_map.items():
        print(f"  [{vname.upper()}] Building QAT model "
              f"(INT{num_bits})...")
        qat_model = model_builder_fn(
            **build_kwargs,
            num_bits = num_bits)
        dummy = np.zeros((1,) + build_kwargs['input_shape'],dtype = np.float32)
        _ = qat_model(dummy, training=False)
        train_qat_model(
            qat_model    = qat_model,
            fp32_model   = fp32_model,
            model_tag    = f'{model_tag}_{vname}',
            train_data   = train_data,
            train_labels = train_labels,
            val_data     = val_data,
            val_labels   = val_labels,
            seed         = calib_seed,
            epochs       = epochs,
            batch_size   = batch_size)

        print(f"\n  [Step 2] Exporting QAT model to ONNX FP32")
        fp32_onnx_path = paths[vname]['fp32_onnx']
        qat_fp32_size = export_to_onnx( 
            keras_model = qat_model,
            save_path   = fp32_onnx_path,
            opset       = OPSET)
        sizes[f'{vname}_fp32_onnx'] = qat_fp32_size
        print(f"  QAT ONNX FP32 size: "
              f"{qat_fp32_size:.4f} MB")
        print(f" Original FP32 ref : "
              f"{fp32_ref_size:.4f} MB")
        print(f"   Size difference   : "
              f"{abs(qat_fp32_size - fp32_ref_size):.4f} MB "
              f"(should be small, ~0.0-0.5 MB)")
        print(f"\n  [Step 3] Running PTQ on QAT ONNX "
              f"({quant_type_map[vname].upper()})...")
        quant_onnx_path = paths[vname]['quant_onnx']
        size_mb = run_static_ptq( 
            fp32_path          = fp32_onnx_path,
            save_path          = quant_onnx_path,
            calibration_data   = calibration_data,
            calibration_labels = calibration_labels,
            quant_type         = quant_type_map[vname],
            calib_seed         = calib_seed,
            n_calib_samples    = N_CALIB_SAMPLES)
        sizes[vname] = size_mb
        reduction_vs_fp32 = ( 1 - size_mb / fp32_ref_size) * 100
        print(f"\n  {vname} quantized size : "
              f"{size_mb:.4f} MB")
        print(f"  Reduction vs FP32 ONNX: "
              f"{reduction_vs_fp32:.1f}%  "
              f"← this is your paper number")

        del qat_model
        tf.keras.backend.clear_session()
        gc.collect()
    print(f"\n  Size Summary ({model_tag} QAT):")
    print(f"  {'Variant':<22}"
          f"{'Size(MB)':>10}"
          f"{'vs FP32 ONNX':>14}"
          f"  Note")
    print(f"  {'fp32_onnx (reference)':<22}"
          f"{fp32_ref_size:>10.4f}"
          f"{'—':>14}"
          f"  baseline (same as PTQ reference)")
    notes = {
        'qat_int8' : 'QAT fine-tuned → PTQ INT8',
        'qat_int16': 'QAT fine-tuned → PTQ INT16',}
    for vname in ['qat_int8', 'qat_int16']:
        vsize = sizes.get(vname, 0)
        if vsize > 0:
            red     = (1 - vsize / fp32_ref_size) * 100
            red_str = f"{red:.1f}%"
        else:
            red_str = 'FAILED'
        print(f"  {vname:<22}"
              f"{vsize:>10.4f}"
              f"{red_str:>14}"
              f"  {notes[vname]}")

    for vname in ['qat_int8', 'qat_int16']:
        results[vname] = {}
        quant_path = paths[vname]['quant_onnx']
        print(f"\n  [{vname}]")

        for ts_name, (ts_data, ts_labels) in \
                test_sets_dict.items():
            results[vname][ts_name] = evaluate_onnx(  
                onnx_path   = quant_path,
                test_data   = ts_data,
                test_labels = ts_labels,
                label_name  = f"{vname}|{ts_name}",
                n_warmup    = 5)
    results['fp32_reference_size_mb'] = fp32_ref_size
    return results

In [ ]:
# Main Loop
all_qat_m1 = {}
all_qat_m2 = {}
for seed in SEEDS:
    print(f"  QAT SEED {seed}  "
          f"({SEEDS.index(seed)+1}/{len(SEEDS)})")
    train_s, val_s, label_s, label_val = (
        train_test_split(
            data_train, label_train,
            test_size    = 0.2,
            shuffle      = True,
            random_state = seed))
    rng       = np.random.RandomState(seed)
    calib_idx = rng.choice(
        len(train_s),
        size    = min(N_CALIB_SAMPLES * 3,
                      len(train_s)),
        replace = False)
    calib_data   = train_s[calib_idx]
    calib_labels = label_s[calib_idx]

    #  MODEL 1 — CNN-RNN
    print(f"\n  [model_1 QAT] Loading FP32 seed {seed}...")
    m1_path = os.path.join( SAVE_DIR, f'model_1_seed{seed}.keras')
    if not os.path.exists(m1_path):
        print(f"  [ERROR] Not found: {m1_path}")
        print(f"  [SKIP]  model_1 QAT seed {seed}")
    else:
        fp32_model_1 = tf.keras.models.load_model(m1_path)
        print(f"  [✓] FP32 model_1 seed {seed} loaded")
        all_qat_m1[seed] = qat_one_model(
            fp32_model         = fp32_model_1,
            model_tag          = f'model_1_seed{seed}',
            model_builder_fn   = build_qat_model_1,
            build_kwargs       = {
                'input_shape': (36, 20, 20, 1),
                'n_class'    : 6},
            train_data         = train_s,
            train_labels       = label_s,
            val_data           = val_s,
            val_labels         = label_val,
            calibration_data   = calib_data,
            calibration_labels = calib_labels,
            test_sets_dict     = test_sets_dict,
            calib_seed         = seed)

        del fp32_model_1
        tf.keras.backend.clear_session()
        gc.collect()
        print(f" model_1 QAT seed {seed} done.")

    #  MODEL 2 — Transformer
    print(f"\n  [model_2 QAT] Loading FP32 seed {seed}...")
    m2_path = os.path.join(SAVE_DIR, f'model_2_seed{seed}.weights.h5')
    if not os.path.exists(m2_path):
        print(f"  [ERROR] Not found: {m2_path}")
        print(f"  [SKIP]  model_2 QAT seed {seed}")
    else:
        fp32_model_2 = transformer_model(
            input_shape = (36, 20, 20, 1),
            n_class     = 6,
            num_layers  = 2)
        dummy = np.zeros((1, 36, 20, 20, 1), dtype=np.float32)
        _ = fp32_model_2(dummy, training=False)
        fp32_model_2.load_weights(m2_path)
        print(f"  [✓] FP32 model_2 seed {seed} loaded")
        all_qat_m2[seed] = qat_one_model(
            fp32_model         = fp32_model_2,
            model_tag          = f'model_2_seed{seed}',
            model_builder_fn   = build_qat_model_2,
            build_kwargs       = {
                'input_shape': (36, 20, 20, 1),
                'n_class'    : 6,
                'num_layers' : 2
            },
            train_data         = train_s,
            train_labels       = label_s,
            val_data           = val_s,
            val_labels         = label_val,
            calibration_data   = calib_data,
            calibration_labels = calib_labels,
            test_sets_dict     = test_sets_dict,
            calib_seed         = seed)

        del fp32_model_2
        tf.keras.backend.clear_session()
        gc.collect()
        print(f" model_2 QAT seed {seed} done.")

    print(f"\n  QAT Seed {seed} complete.")


# Aggregate 
print("  QAT STEP 3: Aggregating results")
aggregated_qat_m1 = aggregate_ptq_across_seeds(all_qat_m1)
aggregated_qat_m2 = aggregate_ptq_across_seeds(all_qat_m2)
print("  QAT STEP 4: Statistical significance tests")
stat_qat_m1 = {}
stat_qat_m2 = {}
for ts_name in ['all_rooms', 'room2', 'room3']:
    print(f"  Model 1 QAT — CNN-RNN | {ts_name}")
    stat_qat_m1[ts_name] = run_statistical_tests(
        all_seed_results = all_qat_m1,
        model_name       = 'Model 1 QAT — CNN-RNN',
        ts_name          = ts_name)
    print(f"  Model 2 QAT — Transformer | {ts_name}")
    stat_qat_m2[ts_name] = run_statistical_tests(
        all_seed_results = all_qat_m2,
        model_name       = 'Model 2 QAT — Transformer',
        ts_name          = ts_name)